In [ ]:
# Colab-ready tone mixer: exports a WAV and downloads it (no playback).
# Usage:
# 1) Run this cell in Google Colab.
# 2) Edit the "USER CONTROLS" to choose which tones are on/off and the duration.
# 3) After it runs, a download prompt will appear for the WAV file.

import numpy as np
from scipy.io.wavfile import write as wav_write
from datetime import datetime

# If running in Colab, this import enables file download
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# -----------------------------
# USER CONTROLS (EDIT THESE)
# -----------------------------
DURATION_SEC = 3600        # length of audio (seconds)
SAMPLE_RATE = 44100        # 44.1 kHz
MASTER_VOLUME = 0.25       # 0.0–1.0 (final peak target after mixing)
APPLY_FADE = True          # apply tiny fade-in/out to avoid clicks

# Turn individual tones on/off
USE_LEFT_528 = False       # Left ear 528 Hz
USE_RIGHT_432 = False     # Right ear 432 Hz
USE_CENTER_10000 = True   # Center (both ears) 10,000 Hz

# Frequencies (Hz) — change if you want different values
LEFT_FREQ_HZ = 528
RIGHT_FREQ_HZ = 432
CENTER_FREQ_HZ = 528

# Optional: add a binaural beat (OFF by default).
# If True, the LEFT/RIGHT toggles & freqs are ignored and replaced by
# a carrier split by +/- BEAT_HZ/2 between ears.
USE_BINAURAL_BEAT = False
BEAT_HZ = 3.0              # e.g., 2–4 Hz for delta (sleep)
CARRIER_HZ = 200.0         # comfortable audible carrier (150–400 Hz is typical)

# Output filename (auto-stamped)
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
WAV_FILENAME = f"tone_mix_{STAMP}.wav"

# -----------------------------
# IMPLEMENTATION
# -----------------------------
def sine_wave(freq_hz, duration, sr):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    return np.sin(2 * np.pi * freq_hz * t)

def apply_fade_io(wave, sr, fade_ms=20):
    """Quick fade-in/out to prevent clicks at start/end."""
    n = len(wave)
    k = int(sr * (fade_ms / 1000.0))
    if k == 0 or k * 2 >= n:
        return wave
    fade_in = np.linspace(0.0, 1.0, k, dtype=wave.dtype)
    fade_out = fade_in[::-1]
    out = wave.copy()
    out[:k] *= fade_in
    out[-k:] *= fade_out
    return out

def normalize_stereo(left, right, target_peak=0.25):
    peak = max(1e-12, float(np.max(np.abs([left, right]))))
    left  = (left  / peak) * target_peak
    right = (right / peak) * target_peak
    return left, right

def build_stereo_mix(duration, sr,
                     use_left, f_left,
                     use_right, f_right,
                     use_center, f_center,
                     use_binaural=False, beat_hz=3.0, carrier_hz=200.0,
                     volume=0.2, apply_fade_flag=True):
    n = int(sr * duration)
    left = np.zeros(n, dtype=np.float32)
    right = np.zeros(n, dtype=np.float32)

    if use_binaural:
        # Left = carrier - beat/2, Right = carrier + beat/2
        fl = max(0.1, carrier_hz - beat_hz / 2.0)
        fr = max(0.1, carrier_hz + beat_hz / 2.0)
        left  += sine_wave(fl, duration, sr).astype(np.float32)
        right += sine_wave(fr, duration, sr).astype(np.float32)
    else:
        if use_left:
            left += sine_wave(f_left, duration, sr).astype(np.float32)
        if use_right:
            right += sine_wave(f_right, duration, sr).astype(np.float32)
        if use_center:
            center = sine_wave(f_center, duration, sr).astype(np.float32)
            left  += center
            right += center

    # Normalize then apply master volume
    left, right = normalize_stereo(left, right, target_peak=volume)

    if apply_fade_flag:
        left = apply_fade_io(left, sr)
        right = apply_fade_io(right, sr)

    stereo = np.column_stack([left, right])  # shape (n, 2)
    return stereo

# -----------------------------
# RUN / EXPORT
# -----------------------------
audio = build_stereo_mix(
    DURATION_SEC, SAMPLE_RATE,
    USE_LEFT_528, LEFT_FREQ_HZ,
    USE_RIGHT_432, RIGHT_FREQ_HZ,
    USE_CENTER_10000, CENTER_FREQ_HZ,
    use_binaural=USE_BINAURAL_BEAT,
    beat_hz=BEAT_HZ,
    carrier_hz=CARRIER_HZ,
    volume=MASTER_VOLUME,
    apply_fade_flag=APPLY_FADE
)

# 16-bit PCM WAV export
wav_int16 = np.int16(np.clip(audio, -1.0, 1.0) * 32767)
wav_write(WAV_FILENAME, SAMPLE_RATE, wav_int16)

print("Export complete.")
print(f"File: {WAV_FILENAME}")
if USE_BINAURAL_BEAT:
    print(f"(Binaural beat ON: {BEAT_HZ} Hz on a {CARRIER_HZ} Hz carrier)")
else:
    print(f"(L:{'On' if USE_LEFT_528 else 'Off'} {LEFT_FREQ_HZ} Hz | "
          f"R:{'On' if USE_RIGHT_432 else 'Off'} {RIGHT_FREQ_HZ} Hz | "
          f"C:{'On' if USE_CENTER_10000 else 'Off'} {CENTER_FREQ_HZ} Hz)")

# Download file
if IN_COLAB:
    try:
        files.download(WAV_FILENAME)
    except Exception as e:
        print("Download failed. You can download the file from the file browser on the left.")
        print("Error:", e)

In [ ]:
# ============================================================
#  FINAL PRODUCTION SCRIPT — 528 Hz + MALE VOICE AFFIRMATION
#  WAV + PDF — Google Colab Ready
# ============================================================

# System + Python deps
!apt-get install -y espeak-ng > /dev/null 2>&1
!pip install -q pydub reportlab matplotlib scipy requests

import numpy as np
import matplotlib.pyplot as plt
from scipy.io.wavfile import write as wav_write
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from datetime import datetime
from pydub import AudioSegment
import subprocess, io, requests
from IPython.display import Audio, display, IFrame

# Detect Colab for auto-download
try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


# ============================================================
# USER SETTINGS
# ============================================================
DURATION_SEC = 600          # total audio length in seconds
SAMPLE_RATE = 44100          # Hz
MASTER_VOLUME = 0.25         # final normalized peak
APPLY_FADE = True            # quick fade in/out on tone

CENTER_FREQ_HZ = 528         # main Solfeggio frequency
AFFIRMATION_TEXT = "You're beautiful!"
AFFIRMATION_INTERVAL = 30.0  # seconds between affirmations

STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
WAV_FILENAME = f"528Hz_HealingSolfeggio_{STAMP}.wav"
PDF_FILENAME = f"528Hz_HealingSolfeggio_Info_{STAMP}.pdf"


# ============================================================
# HELPERS
# ============================================================
def sine_wave(freq_hz, duration, sr):
    """Generate a sine wave at freq_hz for duration seconds."""
    t = np.linspace(0.0, duration, int(sr * duration), endpoint=False)
    return np.sin(2.0 * np.pi * freq_hz * t)


def apply_fade_io(wave, sr, fade_ms=20):
    """Apply short fade-in/out to avoid clicks."""
    n = len(wave)
    k = int(sr * (fade_ms / 1000.0))
    if k < 1 or 2 * k > n:
        return wave
    fade_in = np.linspace(0.0, 1.0, k)
    fade_out = fade_in[::-1]
    wave[:k] *= fade_in
    wave[-k:] *= fade_out
    return wave


def get_latex_image(latex_code):
    """
    Fetch a LaTeX-rendered PNG from Codecogs and wrap as ImageReader
    for use in ReportLab.
    """
    url = f"https://latex.codecogs.com/png.image?\\dpi{{150}} {latex_code}"
    r = requests.get(url)
    r.raise_for_status()
    return ImageReader(io.BytesIO(r.content))


# ============================================================
# 1. MAIN 528 Hz TONE
# ============================================================
tone = sine_wave(CENTER_FREQ_HZ, DURATION_SEC, SAMPLE_RATE).astype(np.float32)
if APPLY_FADE:
    tone = apply_fade_io(tone, SAMPLE_RATE)

audio = tone.copy()  # mono track


# ============================================================
# 2. MALE VOICE AFFIRMATION (eSpeak NG, every 30 seconds)
# ============================================================
def generate_affirmation_wav(text, outfile):
    """
    Use espeak-ng to synthesize a male voice saying `text` into WAV.
    Voice: en+m3 (male), speed 135 (slower).
    """
    subprocess.call([
        "espeak-ng",
        "-v", "en+m3",     # male English voice
        "-s", "135",       # slower speech
        text,
        "-w", outfile
    ])

generate_affirmation_wav(AFFIRMATION_TEXT, "affirmation.wav")

speech_segment = AudioSegment.from_wav("affirmation.wav")
speech_segment = speech_segment.set_frame_rate(SAMPLE_RATE).set_channels(1)
speech_samples = np.array(speech_segment.get_array_of_samples()).astype(np.float32)

# Normalize speech to [-1, 1]
speech_samples /= (np.max(np.abs(speech_samples)) + 1e-9)

# Build speech track across full duration
speech_track = np.zeros_like(audio)
pos = 0.0
while pos < DURATION_SEC:
    start_idx = int(pos * SAMPLE_RATE)
    end_idx = start_idx + len(speech_samples)
    if start_idx >= len(speech_track):
        break
    # Clip if the affirmation would run beyond the end
    slice_len = max(0, min(len(speech_samples), len(speech_track) - start_idx))
    if slice_len > 0:
        speech_track[start_idx:start_idx + slice_len] = speech_samples[:slice_len]
    pos += AFFIRMATION_INTERVAL

# Mix tone and speech
speech_volume = 0.40  # scale the voice relative to the tone
audio = audio + speech_volume * speech_track

# Normalize whole mix to MASTER_VOLUME
peak = float(np.max(np.abs(audio)))
if peak > 0:
    audio = (audio / peak) * MASTER_VOLUME


# ============================================================
# 3. EXPORT WAV + INLINE PLAYBACK + AUTO-DOWNLOAD
# ============================================================
wav_int16 = (audio * 32767.0).astype(np.int16)
wav_write(WAV_FILENAME, SAMPLE_RATE, wav_int16)

print("Exported WAV:", WAV_FILENAME)
display(Audio(WAV_FILENAME))

if IN_COLAB:
    files.download(WAV_FILENAME)


# ============================================================
# 4. PDF GENERATOR
# ============================================================
def generate_pdf(filename, freq, amplitude, sr):
    # ---- Waveform image (first 5 ms) ----
    short_ms = 0.005
    t = np.linspace(0.0, short_ms, int(sr * short_ms), endpoint=False)
    wave = amplitude * np.sin(2.0 * np.pi * freq * t)

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.plot(t * 1000.0, wave, color="#8B0000", linewidth=2)
    ax.set_title(f"{freq} Hz Waveform (first 5 ms)")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.grid(True)

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    waveform_img = ImageReader(buf)

    # ---- LaTeX equation images ----
    general_eq = get_latex_image("s[n] = A\\sin(2\\pi f \\frac{n}{F_s})")
    specific_eq = get_latex_image(
        f"s[n] = {amplitude:.2f}\\sin(2\\pi \\cdot {freq} \\cdot n / {sr})"
    )

    # ---- Start PDF canvas ----
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter
    y = height - 60

    # TITLE
    c.setFont("Helvetica-Bold", 18)
    c.drawString(60, y, f"{freq} Hz Healing Solfeggio Tone")
    y -= 35

    # LEAD-IN RESEARCH PARAGRAPH
    c.setFont("Helvetica", 10)
    lead_in = [
        "Research suggests the 528 Hz audio frequency may influence stress-",
        "related biomarkers, including lowering cortisol, the body's primary",
        "stress hormone, and increasing oxytocin, the body’s primary hormone",
        "for bonding, trust, and social connection <3"

    ]
    for line in lead_in:
        c.drawString(60, y, line)
        y -= 14

    y -= 20

    # EQUATION SECTION
    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "Equation:")
    y -= 18

    c.setFont("Helvetica", 10)
    c.drawString(60, y, "The general equation for a pure tone sine wave (one single frequency) is defined as")
    y -= 25

    # General equation image (small, left-aligned)
    c.drawImage(general_eq, 60, y - 25, width=150, preserveAspectRatio=True)
    y -= 70

    # WHERE SECTION
    c.setFont("Helvetica", 10)
    c.drawString(60, y, "Where:")
    y -= 16

    c.setFont("Helvetica", 10)
    c.drawString(80, y, f"A  = amplitude = {amplitude}")
    y -= 16
    c.drawString(80, y, "f  = frequency in Hz")
    y -= 16
    c.drawString(80, y, "n  = sample index")
    y -= 16
    c.drawString(80, y, f"F_s = sample rate = {sr}")
    y -= 28

    # WORKED EQUATION SECTION
    c.setFont("Helvetica", 10)
    c.drawString(60, y, "Worked Example:")
    y -= 20

    c.drawImage(specific_eq, 60, y - 25, width=200, preserveAspectRatio=True)
    y -= 80

    # WAVEFORM VISUALIZATION SECTION
    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "Waveform Visualization:")
    y -= 12

    c.drawImage(waveform_img, 60, y - 120, width=400, height=120)
    y -= 150

    # REFERENCES SECTION
    c.setFont("Helvetica-Bold", 12)
    c.drawString(60, y, "References:")
    y -= 20

    c.setFont("Helvetica", 10)
    refs = [
        "Fukushima et al. (2018). Effects of 528 Hz on endocrine and mood states.",
        "URL: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5871151/",
        "",
        "Hirokawa et al. (2015). Rat stress markers under 528 Hz.",
        "URL: https://www.jstage.jst.go.jp/article/jvms/77/7/77_150015/_article"
    ]
    for line in refs:
        c.drawString(60, y, line)
        y -= 14

    c.save()


# ============================================================
# 5. CREATE PDF + INLINE PREVIEW + AUTO-DOWNLOAD
# ============================================================
generate_pdf(PDF_FILENAME, CENTER_FREQ_HZ, MASTER_VOLUME, SAMPLE_RATE)

print("Exported PDF:", PDF_FILENAME)
display(IFrame(PDF_FILENAME, width=600, height=800))

if IN_COLAB:
    files.download(PDF_FILENAME)